In [3]:
import duckdb
con = duckdb.connect("../data/retail.duckdb", read_only=True)
con.sql("SELECT * FROM raw_transactions LIMIT 10").show()

┌─────────┬────────────┬─────────────────────────────────────┬──────────┬─────────────────────┬────────┬─────────────┬────────────────┐
│ invoice │ stock_code │             description             │ quantity │    invoice_date     │ price  │ customer_id │    country     │
│ varchar │  varchar   │               varchar               │  int64   │      timestamp      │ double │   double    │    varchar     │
├─────────┼────────────┼─────────────────────────────────────┼──────────┼─────────────────────┼────────┼─────────────┼────────────────┤
│ 489434  │ 85048      │ 15CM CHRISTMAS GLASS BALL 20 LIGHTS │       12 │ 2009-12-01 07:45:00 │   6.95 │     13085.0 │ United Kingdom │
│ 489434  │ 79323P     │ PINK CHERRY LIGHTS                  │       12 │ 2009-12-01 07:45:00 │   6.75 │     13085.0 │ United Kingdom │
│ 489434  │ 79323W     │  WHITE CHERRY LIGHTS                │       12 │ 2009-12-01 07:45:00 │   6.75 │     13085.0 │ United Kingdom │
│ 489434  │ 22041      │ RECORD FRAME 7" SINGLE 

### Q1: What data range does the data cover? 

In [3]:
con.sql("""SELECT MIN(invoice_date) AS first_date,
        MAX(invoice_date) AS last_date
        FROM raw_transactions""").show()

┌─────────────────────┬─────────────────────┐
│     first_date      │      last_date      │
│      timestamp      │      timestamp      │
├─────────────────────┼─────────────────────┤
│ 2009-12-01 07:45:00 │ 2011-12-09 12:50:00 │
└─────────────────────┴─────────────────────┘



### Q2: How many rows have no customer_id?

In [ ]:
con.sql(""" SELECT COUNT (*) AS total_rows,
        COUNT(*) - COUNT(customer_id) AS null_customer_rows,
        ROUND(100.0 * (COUNT(*) - COUNT(customer_id)) / COUNT(*), 1) AS pct_null
        FROM raw_transactions""").show()

┌────────────┬────────────────────┬──────────┐
│ total_rows │ null_customer_rows │ pct_null │
│   int64    │       int64        │  double  │
├────────────┼────────────────────┼──────────┤
│    1067371 │             243007 │     22.8 │
└────────────┴────────────────────┴──────────┘



Based on the table, 243,007 rows (22.8%) have no customer ID. These are likely due to guest checkouts where customers buy without creating an account. These rows still record real sales, so they count toward revenue, product and country analysis. However, they can't be used for RFM segmentation or cohort retention because both need to know who bought. I will keep them for revenue, product and country analysis but I will exclude them from customer-level analysis. 

### Q3: Some invoice numbers start with a letter instead of a number. Which letters, and what do you think they mean?

In [18]:
con.sql("""SELECT LEFT(invoice, 1) AS invoice_first_char,
        COUNT(*) AS invoice_rows,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM raw_transactions), 3) AS pct_of_all_rows
        FROM raw_transactions
        WHERE NOT regexp_matches(invoice, '^[0-9]')
        GROUP BY LEFT(invoice, 1)
        ORDER BY invoice_first_char""").show()

┌────────────────────┬──────────────┬─────────────────┐
│ invoice_first_char │ invoice_rows │ pct_of_all_rows │
│      varchar       │    int64     │     double      │
├────────────────────┼──────────────┼─────────────────┤
│ A                  │            6 │           0.001 │
│ C                  │        19494 │           1.826 │
└────────────────────┴──────────────┴─────────────────┘



In [24]:
con.sql("""SELECT *
        FROM raw_transactions
        WHERE invoice LIKE 'A%'
        LIMIT 10""").show()

┌─────────┬────────────┬─────────────────┬──────────┬─────────────────────┬───────────┬─────────────┬────────────────┐
│ invoice │ stock_code │   description   │ quantity │    invoice_date     │   price   │ customer_id │    country     │
│ varchar │  varchar   │     varchar     │  int64   │      timestamp      │  double   │   double    │    varchar     │
├─────────┼────────────┼─────────────────┼──────────┼─────────────────────┼───────────┼─────────────┼────────────────┤
│ A506401 │ B          │ Adjust bad debt │        1 │ 2010-04-29 13:36:00 │ -53594.36 │        NULL │ United Kingdom │
│ A516228 │ B          │ Adjust bad debt │        1 │ 2010-07-19 11:24:00 │ -44031.79 │        NULL │ United Kingdom │
│ A528059 │ B          │ Adjust bad debt │        1 │ 2010-10-20 12:04:00 │ -38925.87 │        NULL │ United Kingdom │
│ A563185 │ B          │ Adjust bad debt │        1 │ 2011-08-12 14:50:00 │  11062.06 │        NULL │ United Kingdom │
│ A563186 │ B          │ Adjust bad debt │      

In [23]:
con.sql("""SELECT *
        FROM raw_transactions
        WHERE invoice LIKE 'C%'
        LIMIT 10""").show()

┌─────────┬────────────┬───────────────────────────────────┬──────────┬─────────────────────┬────────┬─────────────┬────────────────┐
│ invoice │ stock_code │            description            │ quantity │    invoice_date     │ price  │ customer_id │    country     │
│ varchar │  varchar   │              varchar              │  int64   │      timestamp      │ double │   double    │    varchar     │
├─────────┼────────────┼───────────────────────────────────┼──────────┼─────────────────────┼────────┼─────────────┼────────────────┤
│ C489449 │ 22087      │ PAPER BUNTING WHITE LACE          │      -12 │ 2009-12-01 10:33:00 │   2.95 │     16321.0 │ Australia      │
│ C489449 │ 85206A     │ CREAM FELT EASTER EGG BASKET      │       -6 │ 2009-12-01 10:33:00 │   1.65 │     16321.0 │ Australia      │
│ C489449 │ 21895      │ POTTING SHED SOW 'N' GROW SET     │       -4 │ 2009-12-01 10:33:00 │   4.25 │     16321.0 │ Australia      │
│ C489449 │ 21896      │ POTTING SHED TWINE                │  

From invoice column, I found two sets of invoices starting from letters. First is invoices starting with 'A' with 6 rows (0.001%) and C with 19,494 rows (1.826%). The A rows are bad debt adjustments (description: 'Adjust bad debt') and the C rows are due to cancellations supported by negative values in quantity. I will exclude A rows from sales analysis because they are accounting adjustments, not product sales and exclude C rows for analyzing sales, but the orders cancelled should be noted, since the cancellation rate is a useful business metric on its own.

### Q4: How many rows have quantity <= 0 or price <= 0?

In [61]:
con.sql("""SELECT COUNT (*) AS qty_le_zero
        FROM raw_transactions
        WHERE quantity <= 0
        """).show()

con.sql("""SELECT COUNT (*) AS qty_le_zero_not_c
        FROM raw_transactions
        WHERE quantity <= 0 AND
        invoice NOT LIKE 'C%'
        """).show()

┌─────────────┐
│ qty_le_zero │
│    int64    │
├─────────────┤
│       22950 │
└─────────────┘

┌───────────────────┐
│ qty_le_zero_not_c │
│       int64       │
├───────────────────┤
│              3457 │
└───────────────────┘



Subtraction from qty_le_zero and qty_le_zero_not_c gives 19,493 entries. It is 1 less than the total cancellation count. Probably due to a positive quantity value. 

In [60]:
con.sql("""SELECT *
        FROM raw_transactions
        WHERE quantity <= 0 AND
        invoice NOT LIKE 'C%'
        LIMIT 10
        """).show()

┌─────────┬────────────┬─────────────────┬──────────┬─────────────────────┬────────┬─────────────┬────────────────┐
│ invoice │ stock_code │   description   │ quantity │    invoice_date     │ price  │ customer_id │    country     │
│ varchar │  varchar   │     varchar     │  int64   │      timestamp      │ double │   double    │    varchar     │
├─────────┼────────────┼─────────────────┼──────────┼─────────────────────┼────────┼─────────────┼────────────────┤
│ 489464  │ 21733      │ 85123a mixed    │      -96 │ 2009-12-01 10:52:00 │    0.0 │        NULL │ United Kingdom │
│ 489463  │ 71477      │ short           │     -240 │ 2009-12-01 10:52:00 │    0.0 │        NULL │ United Kingdom │
│ 489467  │ 85123A     │ 21733 mixed     │     -192 │ 2009-12-01 10:53:00 │    0.0 │        NULL │ United Kingdom │
│ 489521  │ 21646      │ NULL            │      -50 │ 2009-12-01 11:44:00 │    0.0 │        NULL │ United Kingdom │
│ 489655  │ 20683      │ NULL            │      -44 │ 2009-12-01 17:26:0

The table shows invoices that were not cancellations but have negative quantity values and null values in customer_id. These are orders that were canceled or modified by a staff member.

In [55]:
con.sql("""SELECT *
        FROM raw_transactions
        WHERE invoice LIKE 'C%'
        AND quantity > 0
        """).show()

┌─────────┬────────────┬─────────────┬──────────┬─────────────────────┬────────┬─────────────┬────────────────┐
│ invoice │ stock_code │ description │ quantity │    invoice_date     │ price  │ customer_id │    country     │
│ varchar │  varchar   │   varchar   │  int64   │      timestamp      │ double │   double    │    varchar     │
├─────────┼────────────┼─────────────┼──────────┼─────────────────────┼────────┼─────────────┼────────────────┤
│ C496350 │ M          │ Manual      │        1 │ 2010-02-01 08:24:00 │ 373.57 │        NULL │ United Kingdom │
└─────────┴────────────┴─────────────┴──────────┴─────────────────────┴────────┴─────────────┴────────────────┘



Table shows the 1 cancellation invoice that has a positive quantity value. Based on stock_code (M) and description (Manual), it is a manual adjustment invoice. Since all C rows are already excluded from sales analysis, this row has no impact on revenue. It is a manual adjustment rather than a cancelled product, but at 1 out of 19,494 rows it does not affect the cancellation rate either.

In [68]:
con.sql("""SELECT COUNT (*) AS price_le_zero
        FROM raw_transactions
        WHERE price <= 0
        """).show()

con.sql("""SELECT COUNT (*) AS price_lt_zero
        FROM raw_transactions
        WHERE price < 0
        """).show()

con.sql("""SELECT COUNT (*) AS price_eq_zero
        FROM raw_transactions
        WHERE price = 0
        """).show()

┌───────────────┐
│ price_le_zero │
│     int64     │
├───────────────┤
│          6207 │
└───────────────┘

┌───────────────┐
│ price_lt_zero │
│     int64     │
├───────────────┤
│             5 │
└───────────────┘

┌───────────────┐
│ price_eq_zero │
│     int64     │
├───────────────┤
│          6202 │
└───────────────┘



In [66]:
con.sql("""SELECT *
        FROM raw_transactions
        WHERE price = 0
        LIMIT 10
        """).show()

┌─────────┬────────────┬──────────────┬──────────┬─────────────────────┬────────┬─────────────┬────────────────┐
│ invoice │ stock_code │ description  │ quantity │    invoice_date     │ price  │ customer_id │    country     │
│ varchar │  varchar   │   varchar    │  int64   │      timestamp      │ double │   double    │    varchar     │
├─────────┼────────────┼──────────────┼──────────┼─────────────────────┼────────┼─────────────┼────────────────┤
│ 489464  │ 21733      │ 85123a mixed │      -96 │ 2009-12-01 10:52:00 │    0.0 │        NULL │ United Kingdom │
│ 489463  │ 71477      │ short        │     -240 │ 2009-12-01 10:52:00 │    0.0 │        NULL │ United Kingdom │
│ 489467  │ 85123A     │ 21733 mixed  │     -192 │ 2009-12-01 10:53:00 │    0.0 │        NULL │ United Kingdom │
│ 489521  │ 21646      │ NULL         │      -50 │ 2009-12-01 11:44:00 │    0.0 │        NULL │ United Kingdom │
│ 489655  │ 20683      │ NULL         │      -44 │ 2009-12-01 17:26:00 │    0.0 │        NULL │ 

In [67]:
con.sql("""SELECT *
        FROM raw_transactions
        WHERE price < 0
        """).show()

┌─────────┬────────────┬─────────────────┬──────────┬─────────────────────┬───────────┬─────────────┬────────────────┐
│ invoice │ stock_code │   description   │ quantity │    invoice_date     │   price   │ customer_id │    country     │
│ varchar │  varchar   │     varchar     │  int64   │      timestamp      │  double   │   double    │    varchar     │
├─────────┼────────────┼─────────────────┼──────────┼─────────────────────┼───────────┼─────────────┼────────────────┤
│ A506401 │ B          │ Adjust bad debt │        1 │ 2010-04-29 13:36:00 │ -53594.36 │        NULL │ United Kingdom │
│ A516228 │ B          │ Adjust bad debt │        1 │ 2010-07-19 11:24:00 │ -44031.79 │        NULL │ United Kingdom │
│ A528059 │ B          │ Adjust bad debt │        1 │ 2010-10-20 12:04:00 │ -38925.87 │        NULL │ United Kingdom │
│ A563186 │ B          │ Adjust bad debt │        1 │ 2011-08-12 14:51:00 │ -11062.06 │        NULL │ United Kingdom │
│ A563187 │ B          │ Adjust bad debt │      

To sum up:

Cancellations (C invoices, quantity<=0): 19,493 rows, same as Q3 (for exclusion in sales analysis)

Inventory adjustments (not C, quantity<=0): 3,457 rows. Came from manual inventory adjustments or modifications made by a staff member evidenced by description such as lost, damages, short, mixed, etc and no customer_id values.

Zero-price: 6,202 rows. These are also inventory modifications where descriptions are made by staff members but quantity values were not all <=0. Positive values possibly due to stock corrections or free items.

Negative-price: 5 rows. These invoices were same as the A rows from Q3 where they are described as bad debt adjustments. However, the A rows contain 6 invoices. That 1 invoice have positive price value. 

Decision: For sales analysis, these cancellation orders, inventory and bad debt adjustments invoices must be excluded, while the orders where quantity > 0 and price > 0 must be kept. 